# Maps and grids from a notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/holoviz/lumen/blob/main/examples/notebooks/maps_and_grids.ipynb)

Ask questions of shapes, points and gridded data, and get maps back. The three datasets ship with `hvsampledata`: US states as polygons, earthquakes as latitude and longitude, and an air temperature grid. Answers depend on the model, so your wording and charts may differ from run to run.

In [ ]:
# In Colab, or anywhere Lumen is not installed yet
%pip install -q "lumen[ai-openai,xarray,geo]" "hvsampledata>=0.1.5" lonboard

In [ ]:
import hvsampledata
import lumen

from lumen.ai.llm import Kilo

states = lumen.read({"states": hvsampledata.us_states("geopandas")})  # polygons
quakes = lumen.read({"quakes": hvsampledata.earthquakes("pandas")})  # lat and lon columns
air = lumen.read({"air": hvsampledata.air_temperature("xarray")})  # a lat, lon, time grid

# Kilo serves free models without an account. With an API key set,
# e.g. OPENAI_API_KEY, leave out llm= and Lumen picks that provider.
llm = Kilo()

## Shapes

A table with a geometry column is drawn as a map on a basemap, shaded by the value you ask for.

In [ ]:
income = states.ask("Map median household income by state", llm=llm)
income

`answer.gdf` is the result as a GeoDataFrame, with its coordinate reference system.

In [ ]:
print(income.gdf.crs)
income.gdf.head(3)

## Points

Latitude and longitude columns are enough for a map; no geometry column is needed.

In [ ]:
quake_map = quakes.ask("Map the earthquakes, coloured by magnitude", llm=llm)
quake_map

## Grids

A gridded dataset is drawn over coastlines. `answer.ds` gives the result back as an xarray Dataset.

In [ ]:
mean_air = air.ask("Map the mean air temperature in degrees Celsius", llm=llm)
mean_air

In [ ]:
mean_air.ds

## Take the results with you

`to_file` picks the format from the extension: GeoParquet, GeoJSON or GeoPackage for shapes, Zarr or NetCDF for grids. `map()` opens the shapes in lonboard when it is installed.

In [ ]:
print(income.to_file("income.parquet"), mean_air.to_file("mean_air.nc"))
income.map()

## Change the chart

Open an answer's Spec section and press Edit to change the chart by hand, then Redraw. `answer.with_spec(...)` does the same from code, with no LLM call.